# 03 Feature Engineering

Experimenting with derived features and validating that they carry real predictive signal before locking them into `src/preprocess.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

df = pd.read_csv('../data/raw/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print('Base shape:', df.shape)

## Feature 1: `avg_monthly_spend`

`TotalCharges / (tenure + 1)` normalises spend by how long the customer has been active.  
Raw `TotalCharges` favours long-tenure customers; this makes churners comparable regardless of when they joined.

In [ ]:
df['avg_monthly_spend'] = df['TotalCharges'] / (df['tenure'] + 1)

# Compare distributions
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for churn_val, label, color in [(0, 'Retained', 'steelblue'), (1, 'Churned', 'tomato')]:
    sub = df[df['Churn'] == churn_val]
    axes[0].hist(sub['TotalCharges'],      bins=30, alpha=0.6, label=label, color=color)
    axes[1].hist(sub['avg_monthly_spend'], bins=30, alpha=0.6, label=label, color=color)

axes[0].set_title('Raw TotalCharges')
axes[1].set_title('avg_monthly_spend (engineered)')
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print('Correlation with Churn:')
print(f"  TotalCharges     : {df['TotalCharges'].corr(df['Churn']):.4f}")
print(f"  avg_monthly_spend: {df['avg_monthly_spend'].corr(df['Churn']):.4f}")

## Feature 2: `num_services`

Count of active add-on services. Hypothesis: customers with fewer services have less lock-in and churn more.

In [ ]:
add_ons = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
]
df['num_services'] = sum((df[s] == 'Yes').astype(int) for s in add_ons)

# Churn rate per service count
churn_by_services = df.groupby('num_services')['Churn'].mean() * 100

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(churn_by_services.index, churn_by_services.values, color='steelblue')
ax.set_xlabel('Number of services subscribed')
ax.set_ylabel('Churn rate (%)')
ax.set_title('Churn Rate by Number of Services')
plt.tight_layout()
plt.show()

print(churn_by_services.round(1))

## Feature 3: `high_value`

Binary flag: 1 if `MonthlyCharges` is above the 75th percentile.

In [ ]:
p75 = df['MonthlyCharges'].quantile(0.75)
df['high_value'] = (df['MonthlyCharges'] > p75).astype(int)

print(f'75th percentile threshold: {p75:.2f}')
print(f'\nChurn rate by high_value flag:')
print(df.groupby('high_value')['Churn'].mean().mul(100).round(1).to_string())

## Feature 4: `tenure_group`

Buckets tenure into business-meaningful bands.

In [ ]:
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-1yr', '1-2yr', '2-4yr', '4-6yr'],
)

churn_by_tenure = df.groupby('tenure_group', observed=True)['Churn'].mean() * 100

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(churn_by_tenure.index.astype(str), churn_by_tenure.values, color='steelblue')
ax.set_ylabel('Churn rate (%)')
ax.set_title('Churn Rate by Tenure Band')
plt.tight_layout()
plt.show()

print(churn_by_tenure.round(1))

## Validation: do engineered features improve ROC-AUC?

Quick logistic regression comparison: baseline features vs baseline + engineered features.

In [ ]:
def quick_auc(X_df, y):
    """Encode, split, scale, fit LR, return ROC-AUC."""
    X = X_df.copy()
    le = LabelEncoder()
    for col in X.select_dtypes(include=['object', 'category']).columns:
        X[col] = le.fit_transform(X[col].astype(str))

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    imp = SimpleImputer(strategy='median')
    sc  = StandardScaler()
    Xtr_sc = sc.fit_transform(imp.fit_transform(Xtr))
    Xte_sc = sc.transform(imp.transform(Xte))

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(Xtr_sc, ytr)
    return roc_auc_score(yte, model.predict_proba(Xte_sc)[:, 1])


y = df['Churn']

# Baseline: raw columns only
base_cols = [c for c in df.columns if c not in
             ['customerID', 'Churn', 'avg_monthly_spend', 'num_services', 'high_value', 'tenure_group']]
auc_base = quick_auc(df[base_cols], y)

# With engineered features
eng_cols  = base_cols + ['avg_monthly_spend', 'num_services', 'high_value', 'tenure_group']
auc_eng   = quick_auc(df[eng_cols], y)

print(f'Baseline ROC-AUC         : {auc_base:.4f}')
print(f'+ Engineered ROC-AUC     : {auc_eng:.4f}')
print(f'Delta                    : +{auc_eng - auc_base:.4f}')

## Feature correlation matrix (engineered + key raw)

In [ ]:
check_cols = ['tenure', 'MonthlyCharges', 'TotalCharges',
              'avg_monthly_spend', 'num_services', 'high_value', 'Churn']

corr_matrix = df[check_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(check_cols)))
ax.set_yticks(range(len(check_cols)))
ax.set_xticklabels(check_cols, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(check_cols, fontsize=8)
for i in range(len(check_cols)):
    for j in range(len(check_cols)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

## Summary

| Feature | Signal | Decision |
|---|---|---|
| `avg_monthly_spend` | Highest correlation with churn | Keep |
| `num_services` | Clear monotonic churn decrease as services increase | Keep |
| `high_value` | Modest uplift | Keep (low cost) |
| `tenure_group` | Strong churn gradient across bands | Keep |

All four features included in `src/preprocess.py to engineer_features`.